# RSNA Knee: reproducible 2.5D baseline

Attach the competition dataset and the `labels-final` Kaggle Dataset. That second dataset must contain `rsna_knee_pipeline.py`, `labels_final.csv`, and `labels_research.csv` in its root; `labels_llm_test.csv` is optional. The first cell reads those label files from `/kaggle/input/datasets/bokkyn/labels-final/` (change `LABEL_DATASET_DIR` if Kaggle mounts your dataset elsewhere). Select **GPU T4** in Notebook options. The first run audits metadata and labels, then runs a small GOLD-only smoke test. Kaggle input is read-only; outputs go to `/kaggle/working/`.

In [ ]:
from pathlib import Path
import os, sys

INPUT_ROOT = Path('/kaggle/input')
COMPETITION_DIR = INPUT_ROOT / 'competitions' / 'rsna-knee-abnormality-detection'
LABEL_DATASET_DIR = Path('/kaggle/input/datasets/bokkyn/labels-final')
OUTPUT_DIR = Path('/kaggle/working')
RUN_FULL_GOLD_BASELINE = False
USE_SILVER = False
USE_QWEN_SOFT_LABELS = False
RESUME_FROM_CHECKPOINT = None

if not INPUT_ROOT.is_dir():
    raise FileNotFoundError('Kaggle inputs are unavailable at /kaggle/input. Attach the competition data and project dataset first.')
if not COMPETITION_DIR.is_dir():
    available_competitions = list((INPUT_ROOT / 'competitions').iterdir()) if (INPUT_ROOT / 'competitions').is_dir() else []
    raise FileNotFoundError(f'Competition directory not found: {COMPETITION_DIR}. Available: {available_competitions}. Update COMPETITION_DIR to the directory containing train.csv.')
if not LABEL_DATASET_DIR.is_dir():
    conventional_label_dir = INPUT_ROOT / 'labels-final'
    if conventional_label_dir.is_dir():
        LABEL_DATASET_DIR = conventional_label_dir
    else:
        raise FileNotFoundError(f'Label/project dataset directory not found: {LABEL_DATASET_DIR}. Attach the labels-final dataset or set LABEL_DATASET_DIR to its Kaggle input path.')
pipeline_path = LABEL_DATASET_DIR / 'rsna_knee_pipeline.py'
if not pipeline_path.is_file():
    raise FileNotFoundError(f'Missing {pipeline_path}. Upload rsna_knee_pipeline.py into the same Kaggle Dataset as labels_final.csv and labels_research.csv.')
sys.path.insert(0, str(LABEL_DATASET_DIR))
import rsna_knee_pipeline as knee

knee.setup_logging()

inputs = knee.discover_inputs(COMPETITION_DIR)
inputs['parser_csv'] = LABEL_DATASET_DIR / 'labels_final.csv'
inputs['parser_research_csv'] = LABEL_DATASET_DIR / 'labels_research.csv'
optional_qwen = LABEL_DATASET_DIR / 'labels_llm_test.csv'
inputs['qwen_csv'] = optional_qwen if optional_qwen.is_file() else None
for label_name in ('parser_csv', 'parser_research_csv'):
    if not inputs[label_name].is_file():
        raise FileNotFoundError(f"Missing required label CSV: {inputs[label_name]}")
required_files = ('train_csv', 'train_series_csv', 'test_series_csv', 'sample_submission')
missing_files = [name for name in required_files if inputs[name] is None or not inputs[name].is_file()]
if missing_files:
    raise FileNotFoundError(f'Missing required Kaggle CSV inputs: {missing_files}. Attach the complete competition dataset.')
if not inputs['train_image_roots']:
    raise FileNotFoundError('Could not find the training DICOM directory. Expected a train_images/train directory under /kaggle/input.')
if not inputs['test_image_roots']:
    raise FileNotFoundError('Could not find the test DICOM directory. Expected a test_images/test directory under /kaggle/input.')
print('Competition CSV inputs:', {key: str(inputs[key]) for key in required_files})
print('Label CSV inputs:', {key: str(inputs[key]) if inputs[key] else None for key in ('parser_csv', 'parser_research_csv', 'qwen_csv')})
print('Training image roots:', inputs['train_image_roots'])
print('Test image roots:', inputs['test_image_roots'])
knee._system_report()

## Audit and smoke test

The label audit does not load pixel payloads. The training/smoke-test step performs one DICOM metadata scan and creates DICOM, series-selection, and project reports in the working directory. The smoke test decodes selected DICOMs, checks a GOLD-only split and loss, trains for at most two epochs/three batches, and writes `smoke_test_report.txt`.

In [ ]:
knee.run_audit(inputs, OUTPUT_DIR, audit_dicom=False)
smoke_config = knee.PipelineConfig(
    smoke_test=True,
    gold_only=True,
    use_silver=False,
    use_qwen_soft_labels=False,
    pretrained=False,
    image_size=320,
    slices_per_study=3,
    batch_size=2,
    num_workers=0,
    num_epochs=2,
    max_studies=64,
    max_batches_per_epoch=3,
    seed=42,
)
smoke_report = knee.run_smoke_test(smoke_config, inputs, OUTPUT_DIR)
assert smoke_report['status'] == 'PASS', smoke_report

## GOLD-only baseline

Run only after the smoke test passes. The pretrained ResNet34 weights must be available to the Kaggle runtime (internet enabled or torchvision weights cached). The validation macro-AUC averages only targets that have both GOLD classes; unavailable target AUCs remain `NaN`. Use the same seed and split when comparing later experiments.

In [ ]:
if RUN_FULL_GOLD_BASELINE:
    baseline_config = knee.PipelineConfig(
        gold_only=not (USE_SILVER or USE_QWEN_SOFT_LABELS),
        use_silver=USE_SILVER,
        use_qwen_soft_labels=USE_QWEN_SOFT_LABELS,
        pretrained=True,
        image_size=320,
        slices_per_study=5,
        batch_size=4,
        num_workers=2,
        num_epochs=10,
        seed=42,
        cache_images=True,
    )
    resume_path = Path(RESUME_FROM_CHECKPOINT) if RESUME_FROM_CHECKPOINT else None
    training_report = knee.train_model(baseline_config, inputs, OUTPUT_DIR, resume_path)
    print(training_report)
else:
    print('Full baseline not started. Set RUN_FULL_GOLD_BASELINE = True after reviewing the smoke report.')

## Optional test inference

Submission generation requires the official `sample_submission.csv`, test image directory, and a trained checkpoint. The code checks exact IDs/columns/order/score range and fails rather than inventing an output format. Save all desired outputs from `/kaggle/working/` as a notebook version or Kaggle Dataset.

In [ ]:
MAKE_SUBMISSION = False
if MAKE_SUBMISSION:
    if not (OUTPUT_DIR / 'best_model.pth').is_file():
        raise FileNotFoundError('Train a model and save best_model.pth before inference.')
    submission_path = knee.generate_submission(OUTPUT_DIR / 'best_model.pth', inputs, OUTPUT_DIR)
    print('Validated submission:', submission_path)